# Shift-Scan Research Notebook

**Power-of-two state dynamics, ternary projections, state-aware distillation, and HARPQ (Horizon-Aware Retention Quantization).**

This notebook runs the actual project code in the accompanying ZIP. It uses a pretrained Mamba-2 checkpoint and streamed FineWeb-Edu documents for calibration/training/evaluation. Random tensors are restricted to deterministic unit tests and kernel microbenchmarks.

**Important:** a GPU runtime is required for model loading, calibration, QAT, and inference benchmarks. Start with a small token budget to validate the pipeline; use matched token budgets and multiple seeds for research results. Do not report unexecuted results.


## 1. Upload and unpack the project ZIP

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os

uploaded = files.upload()  # Select shift_scan_research_upgrade.zip
zip_name = next((n for n in uploaded if n.endswith('.zip')), None)
assert zip_name is not None, 'Upload the Shift-Scan project ZIP.'
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('/content')
PROJECT = Path('/content/shift_scan')
os.chdir(PROJECT)
print('Project:', PROJECT)
print('Files:', sorted(p.name for p in PROJECT.iterdir() if p.is_file()))


## 2. Install dependencies and verify the runtime

In [ ]:
!pip -q install -r requirements.txt


In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA runtime:', torch.version.cuda)
else:
    raise RuntimeError('Switch Colab to Runtime > Change runtime type > GPU before running model experiments.')


## 3. Run deterministic correctness tests (CPU/GPU independent)

In [ ]:
!python tests.py


## 3b. Prior-distribution analysis suite (CPU, ~1 min)

Five controlled studies: decay-grid selection (nearest vs HARPQ vs **dyadic two-term**), fixed-point **dead zone**, ternary scale rules, recurrent error-bound tightness, packed-size accounting. These use the Mamba-2 *initialisation prior* and exact arithmetic, **not trained-checkpoint results**; the real-checkpoint counterpart is Section 4.

In [ ]:
!python research_suite.py --out results/suite
from IPython.display import Image, display
for f in ['s1_decay_grid','s2_deadzone','s3_ternary','s4_bound']:
    display(Image(f'results/suite/{f}.png'))


## 4. Calibrate decay on real held-out FineWeb-Edu text

This compares nearest one-step leak quantization with the proposed HARPQ finite-horizon retention objective. Calibration data are selected from the deterministic validation-document shard. The result is a coefficient diagnostic, not a language-model benchmark.

In [ ]:
!mkdir -p results
!python calibrate_decay.py \
  --pretrained state-spaces/mamba2-130m \
  --dataset HuggingFaceFW/fineweb-edu --subset sample-10BT \
  --tokens 200000 --seq-len 1024 --horizon 128 --weight log \
  --out results/decay_calibration.json


## 5. Train the joint Shift-Scan student on real data

The default below is a full 100M-token experiment and may take hours. First use `--tokens 1000000` as a smoke test, confirm that the model loads and the loss decreases, then launch the full matched-budget run. A T4 may need `--dtype fp16 --micro-bs 1 --accum 32`; an A100 can usually use larger batches. Adjust for actual GPU memory.

In [ ]:
!python train_qat.py \
  --name shift_scan_harpq_main \
  --pretrained state-spaces/mamba2-130m \
  --dataset HuggingFaceFW/fineweb-edu --subset sample-10BT \
  --tokens 100000000 --seq-len 2048 \
  --decay shift --decay-quantizer horizon --decay-horizon 128 --decay-lag-weight uniform \
  --decay-group-size 1 --ternary 1 --group-size 128 \
  --activation-aware 1 --state-weight 0.05 --state-loss-type hybrid \
  --dtype bf16 --micro-bs 2 --accum 8 \
  --seed 0 --out runs


### 5b. Dyadic (two-term) decay variant

Same budget as the main run, only the decay grid changes (`--decay-quantizer dyadic`, 105 levels instead of 14). Decode runs on the simulated path: no integer-state kernel exists yet for two shifts.

In [ ]:
!python train_qat.py \
  --name shift_scan_dyadic2 \
  --pretrained state-spaces/mamba2-130m \
  --dataset HuggingFaceFW/fineweb-edu --subset sample-10BT \
  --tokens 100000000 --seq-len 2048 \
  --decay shift --decay-quantizer dyadic --dyadic-terms 2 --decay-horizon 128 \
  --ternary 1 --group-size 128 --activation-aware 1 \
  --state-weight 0.05 --state-loss-type hybrid \
  --dtype bf16 --micro-bs 2 --accum 8 --seed 0 --out runs


## 6. Evaluate language modeling and long-context behavior

In [ ]:
!python eval_lm.py \
  --ckpt runs/shift_scan_harpq_main/last.pt \
  --tasks lambada_openai,hellaswag,piqa,arc_easy,arc_challenge,winogrande,wikitext \
  --ppl-length --out results/shift_scan_harpq_main.json


## 7. Benchmark actual decode performance

This requires CUDA and a trained checkpoint. The benchmark separates simulated fake-quant PyTorch execution from packed ternary/Triton execution. Analytic energy estimates are not measured energy; use `--nvml` only when NVML is available and report sampling limitations.

In [ ]:
!python bench.py \
  --ckpt runs/shift_scan_harpq_main/last.pt \
  --batches 1,8 --out results/shift_scan_harpq_main_bench.json


In [ ]:
# Optional: true int32 fixed-point state + integer right-shift decay in decode.
# This still uses FP32 state injection/readout. It is experimental and requires GPU validation.
!python bench.py --ckpt runs/shift_scan_harpq_main/last.pt --integer-state \
  --state-frac-bits 20 --out results/shift_scan_integer_state_bench.json


## 8. Run the matched ablation matrix

The command below prints the complete experiment commands without launching them. Add `--run` to execute sequentially. For publication-quality conclusions, run each important condition with at least three seeds and identical data/token budgets.

In [ ]:
!python run_ablation.py --tokens 100000000 --seq-len 2048 --out-dir runs/ablation --print
# To execute all jobs, run the following in a separate cell after confirming GPU/time budget:
# !python run_ablation.py --tokens 100000000 --seq-len 2048 --out-dir runs/ablation --run


## 9. Aggregate and inspect results

In [ ]:
!python analyze_runs.py --runs runs --eval-dir results --out results/ablation_summary.csv
!python retention_analysis.py results/decay_calibration.json --out results/retention_summary.csv


In [ ]:
import pandas as pd
from pathlib import Path
for f in ['results/ablation_summary.csv', 'results/retention_summary.csv']:
    p = Path(f)
    if p.exists():
        print('\n', f)
        display(pd.read_csv(p))


## 10. Research reporting checklist

Before writing claims, record: exact checkpoint revision, tokenizer, dataset/config, held-out document split, tokens seen, sequence length, seed, software versions, GPU model, dtype, HARPQ horizon/lag weights, exponent histograms, retention error by lag, validation perplexity, downstream task scores, packed bytes including scale metadata and FP rows, decode/prefill latency, memory, and measured energy only if actual device telemetry supports it.

**Novelty discipline:** HARPQ is the project's proposed research contribution, not a verified first. Search recent literature and released code for retention-aware quantization and discrete/fixed-point SSM dynamics before making a priority claim. The floating-point PoT path is not an integer bit-shift kernel; the fixed-point file is a correctness reference, not yet a fast integrated inference path.
